# PKG CPTY — Step 0a: payment table profile (Impala)

**Phase 1** runs blind: schema, partitions, column stats, role candidates. Only `TABLE` is needed.
**Guard**: fill `ROLES`, `MONTH_EXPR`, `MONTH_PRED`, `RANGE_PRED` from Phase 1 output, set `ROLES_CONFIRMED = True`.
**Phase 2** (single months): every column — absent rate by rows AND dollars, NDV, domain values of low-cardinality columns.
**Phase 3** (2024-01 → last complete month, role columns only):
  3a volume by month × rail × direction × source · 3b FI routing / FI name / cpty name / cpty id coverage + overlap
  3c beneficiary address coverage · 3d merchant flag: distribution, per-cpty consistency, overlap with cpty degree
  3e trx_id structure by source × rail, within-source duplicate ratio, timestamp granularity

Design decisions:
- **Unfiltered on purpose.** Step 0 profiles the raw table; Neo4j-ingestion filters are applied in 0b once parity is known.
  `BASE_FILTER` exists so the same notebook can be re-run on the filtered population for comparison.
- **Dollars = |amount|.** Sign conventions are profiled separately (3a) — sign may encode direction.
- **Absent = NULL or blank** (after TRIM) for strings; NULL for everything else. Dimension labels keep the
  distinction (`<NULL>` vs `<BLANK>`) because the two usually come from different upstream paths.
- **Routing numbers are classified, not just null-checked**: absent / all-zero sentinel / BIC-like / malformed /
  ABA checksum fail / ABA valid. A populated-but-invalid routing field cannot support a distinct-FI count.
- Numbers and tables inline; charts to disk; every result also written to CSV with its SQL for audit.

In [ ]:
from pathlib import Path
import datetime as dt
import logging
import re
import time

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

import dbi   # Impala helper used in earlier PKG notebooks — adjust the import if it lives elsewhere

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("step0a")
pd.set_option("display.max_rows", 300, "display.max_columns", 60, "display.width", 250)

# ============================== CONFIG ==============================
TABLE       = "<db>.<payment_table>"            # <-- the only thing Phase 1 needs
OUT_DIR     = Path("../metrics/cpty_profile/step0a")
FIG_DIR     = OUT_DIR / "figs"
SQL_DIR     = OUT_DIR / "sql"
DSN, POOL   = "DSN=bdpimp04-impala;", "root.CIB-AMG_Impala"

MONTH_MIN   = "2024-01"                          # pre-2024 excluded as unreliable (standing constraint)
_first      = dt.date.today().replace(day=1)
MONTH_MAX   = (_first - dt.timedelta(days=1)).strftime("%Y-%m")   # last complete month
PROFILE_MONTHS = [MONTH_MIN, MONTH_MAX]          # single-month deep dives (Phase 2, 3d, 3e)

LOWCARD_MAX = 200     # NDV at or below this -> full domain-value table
COL_CHUNK   = 30      # columns per Phase-2 query
BASE_FILTER = "1=1"   # raw on purpose; see header

# ---- Filled from Phase 1 output, then ROLES_CONFIRMED = True ----
ROLES_CONFIRMED = False
ROLES = {
    "amount": None,         # REQUIRED
    "rail": None,           # REQUIRED
    "direction": None, "source_system": None, "trx_id": None,
    "txn_date": None, "txn_ts": None,
    "cust_id": None, "acct_id": None,                    # PNC side (degree uses cust_id, else acct_id)
    "cpty_id": None, "cpty_name": None,
    "fi_routing": None, "fi_name": None,
    "merchant_flag": None, "status": None,
    "address": [],          # every beneficiary / counterparty address column
}
# Month expression and pruning predicates. Placeholders: {m}='YYYY-MM', {y}=YYYY, {mo}=M, {ym}=YYYYMM;
# range: {lo},{hi} ('YYYY-MM') and {lo_ym},{hi_ym} (YYYYMM ints).
MONTH_EXPR = None   # e.g. "substr(cast(trx_dt AS STRING), 1, 7)"
MONTH_PRED = None   # e.g. "part_month = '{m}'"          or "yr = {y} AND mo = {mo}"
RANGE_PRED = None   # e.g. "part_month BETWEEN '{lo}' AND '{hi}'"  or "yr*100+mo BETWEEN {lo_ym} AND {hi_ym}"
# ====================================================================

for d in (OUT_DIR, FIG_DIR, SQL_DIR):
    d.mkdir(parents=True, exist_ok=True)


def q(sql, name=None):
    """Run on Impala; lower-case columns; persist result + SQL when named."""
    t0 = time.time()
    df = dbi.db_get_query(sql, dsn=DSN, pool=POOL, conn_options={"SocketTimeout": 0})
    df.columns = [str(c).lower().strip() for c in df.columns]
    log.info("query %-30s %8d rows %7.1fs", name or "-", len(df), time.time() - t0)
    if name:
        df.to_csv(OUT_DIR / f"{name}.csv", index=False)
        (SQL_DIR / f"{name}.sql").write_text(sql)
    return df


def num(df, cols):
    """ODBC returns DECIMAL as object; coerce measure columns to float."""
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype(float)
    return df

## Phase 1 — schema, partitions, column stats, role candidates

In [ ]:
desc_raw = q(f"DESCRIBE FORMATTED {TABLE}", "p1_describe_raw")


def parse_describe(df):
    cols, parts, info, section = [], [], {}, "cols"
    for name, typ, comment in df.iloc[:, :3].fillna("").astype(str).itertuples(index=False):
        n, t, c = name.strip(), typ.strip(), comment.strip()
        if n.startswith("# Partition Information"):
            section = "parts"; continue
        if n.startswith("# Detailed Table Information") or n.startswith("# Storage Information"):
            section = "info"; continue
        if section in ("cols", "parts"):
            if not n or n.startswith("#"):
                continue
            (cols if section == "cols" else parts).append((n.lower(), t.lower(), c))
        else:
            if n:
                info[n.rstrip(":")] = t
            elif t:                       # table parameters: ('', 'numRows', '123')
                info[t] = c
    return cols, parts, info


data_cols, part_cols, tbl_info = parse_describe(desc_raw)
schema = pd.DataFrame(
    [(c, t, cm, False) for c, t, cm in data_cols] + [(c, t, cm, True) for c, t, cm in part_cols],
    columns=["column", "type", "comment", "is_partition"])
assert len(schema), "DESCRIBE FORMATTED parsed no columns — check TABLE / dbi output shape"

print(f"{TABLE}: {len(data_cols)} data columns, partitioned by {[c for c, _, _ in part_cols] or 'NOTHING'}")
for k in ("Location", "InputFormat", "numRows", "totalSize", "transient_lastDdlTime", "impala.lastComputeStatsTime"):
    if k in tbl_info:
        print(f"  {k:<30} {tbl_info[k]}")

try:
    cstats = q(f"SHOW COLUMN STATS {TABLE}", "p1_column_stats")
    cstats = cstats.rename(columns={"#distinct values": "stats_ndv", "#nulls": "stats_nulls"})
    schema = schema.merge(cstats[["column", "stats_ndv", "stats_nulls"]], on="column", how="left")
    if (pd.to_numeric(schema.stats_ndv, errors="coerce").fillna(-1) < 0).all():
        print("  column stats absent (-1): COMPUTE STATS has not run; Phase 2 measures directly")
except Exception as e:                       # noqa: BLE001 — stats are a convenience, not a dependency
    log.warning("SHOW COLUMN STATS failed: %s", e)

display(schema)

In [ ]:
parts = None
if part_cols:
    parts = q(f"SHOW PARTITIONS {TABLE}", "p1_partitions")
    pk = [c for c, _, _ in part_cols]
    parts = parts[parts[pk[0]].astype(str).str.lower() != "total"]
    keep = pk + [c for c in ("#rows", "#files", "size", "format", "incremental stats") if c in parts.columns]
    print(f"{len(parts)} partitions; first 3 and last 30:")
    display(pd.concat([parts[keep].head(3), parts[keep].tail(30)]))
    if "#rows" in parts.columns and (pd.to_numeric(parts["#rows"], errors="coerce").fillna(-1) < 0).all():
        print("partition #rows = -1 everywhere (no stats) — Phase 3a computes monthly counts")

In [ ]:
ROLE_PATTERNS = {
    "amount":        r"amt|amount",
    "txn_date":      r"(^|_)(dt|date)($|_)|date",
    "txn_ts":        r"(^|_)ts($|_)|tmstmp|timestamp|_time",
    "rail":          r"rail|chnl|channel|pmt_ty|pymt_ty|payment_type|trx_ty|txn_ty|tran_ty|prod",
    "direction":     r"dir|dr_cr|drcr|debit|credit|in_out|flow_ind",
    "source_system": r"src|source|sys_cd|system|feed",
    "trx_id":        r"trx_id|txn_id|tran_id|transaction_id|trace|ref|uetr|e2e|end_to_end|msg_id|imad|omad",
    "cust_id":       r"mdm|cust|pwr_id|party",
    "acct_id":       r"acct|account",
    "cpty_id":       r"cpty|counterparty|cntrpty|bene|payee|payer|orig|rcvr|receiver|sender",
    "cpty_name":     r"(cpty|counterparty|bene|payee|payer|orig|rcvr|receiver|sender|merch).*(nm|name)",
    "fi_routing":    r"aba|routing|rtn|rt_nbr|rt_num|bic|swift|fed_id|chips",
    "fi_name":       r"(bank|bnk|fi_|inst|agent).*(nm|name)",
    "merchant_flag": r"merch|mrch|mcc|cpty_(ind|flg|flag|typ|type)",
    "status":        r"stat|rvrs|revers|rtrn|return|void|cancel|reject",
    "address":       r"addr|street|city|state|st_cd|zip|postal|cntry|country",
}
cand = []
for role, pat in ROLE_PATTERNS.items():
    hits = schema[schema["column"].str.contains(pat, flags=re.I, regex=True)
                  | schema["comment"].str.contains(pat, flags=re.I, regex=True)]
    cand.append((role, ", ".join(f"{c}:{t}" for c, t in zip(hits["column"], hits["type"])) or "—"))
print("Role candidates (name OR comment match) — shortlist only; confirm by hand:")
display(pd.DataFrame(cand, columns=["role", "candidates"]))

## Guard — confirm roles before anything scans the table

In [ ]:
if not ROLES_CONFIRMED:
    raise RuntimeError("Fill ROLES / MONTH_EXPR / MONTH_PRED / RANGE_PRED from Phase 1, "
                       "set ROLES_CONFIRMED = True, and re-run the CONFIG cell.")

TYPES = dict(zip(schema["column"], schema["type"]))
ROLES = {k: ([c.lower() for c in v] if isinstance(v, list) else (v.lower() if v else None)) for k, v in ROLES.items()}
_missing = [c for v in ROLES.values() for c in (v if isinstance(v, list) else [v]) if c and c not in TYPES]
assert not _missing, f"role columns not in schema: {_missing}"
assert ROLES["amount"] and ROLES["rail"], "amount and rail are required"
assert MONTH_EXPR and MONTH_PRED and RANGE_PRED, "MONTH_EXPR / MONTH_PRED / RANGE_PRED required"

STRINGY = ("string", "varchar", "char")
is_str = lambda c: TYPES[c].startswith(STRINGY)
absent = lambda c: f"({c} IS NULL OR TRIM({c}) = '')" if is_str(c) else f"({c} IS NULL)"
AMT_RAW = f"CAST({ROLES['amount']} AS DECIMAL(38,2))"
AMT = f"ABS({AMT_RAW})"


def lbl(col):
    """Dimension label that keeps NULL and blank distinct."""
    if not col:
        return "'(no column)'"
    s = f"CAST({col} AS STRING)"
    return f"CASE WHEN {col} IS NULL THEN '<NULL>' WHEN TRIM({s}) = '' THEN '<BLANK>' ELSE TRIM({s}) END"


def month_pred(m):
    return MONTH_PRED.format(m=m, y=int(m[:4]), mo=int(m[5:7]), ym=int(m[:4] + m[5:7]))


def range_pred():
    return RANGE_PRED.format(lo=MONTH_MIN, hi=MONTH_MAX,
                             lo_ym=int(MONTH_MIN.replace("-", "")), hi_ym=int(MONTH_MAX.replace("-", "")))


def rt_class(c):
    """Routing-number quality class. Numeric storage loses ABA leading zeros (011000015 -> 11000015): re-pad."""
    if not c:
        return "'(no column)'"
    if is_str(c):
        s = f"TRIM(CAST({c} AS STRING))"
    else:
        log.warning("fi_routing %s is %s — leading zeros were destroyed upstream; re-padding to 9", c, TYPES[c])
        s = f"LPAD(CAST({c} AS STRING), 9, '0')"
    d = lambda i: f"CAST(substr({s},{i},1) AS INT)"
    chk = f"pmod(3*({d(1)}+{d(4)}+{d(7)}) + 7*({d(2)}+{d(5)}+{d(8)}) + ({d(3)}+{d(6)}+{d(9)}), 10)"
    return f"""CASE
        WHEN {absent(c)} THEN 'absent'
        WHEN regexp_like({s}, '^0+$') THEN 'sentinel_zero'
        WHEN regexp_like(upper({s}), '^[A-Z]{{6}}[A-Z0-9]{{2}}([A-Z0-9]{{3}})?$') THEN 'bic_like'
        WHEN NOT regexp_like({s}, '^[0-9]{{9}}$') THEN 'malformed'
        WHEN {chk} <> 0 THEN 'aba_checksum_fail'
        ELSE 'aba_valid' END"""


def pct_block(df, by, masks):
    """Totals plus rows% and usd% per group for each boolean mask — coverage always reported both ways."""
    tot = df.groupby(by)[["n", "usd"]].sum()
    out = {"rows_M": tot.n / 1e6, "usd_B": tot.usd / 1e9}
    for lab, m in masks.items():
        h = df[m].groupby(by)[["n", "usd"]].sum().reindex(tot.index, fill_value=0)
        out[f"{lab} rows%"] = 100 * h.n / tot.n
        out[f"{lab} usd%"] = 100 * h.usd / tot.usd.replace(0, np.nan)
    return pd.DataFrame(out).round(2)


# Month-predicate sanity: pruning predicate and MONTH_EXPR must agree, or every later number is mislabelled.
for m in PROFILE_MONTHS:
    chk = num(q(f"SELECT MIN({MONTH_EXPR}) AS lo, MAX({MONTH_EXPR}) AS hi, COUNT(*) AS n "
                f"FROM {TABLE} WHERE {month_pred(m)}"), ["n"]).iloc[0]
    assert chk.n > 0, f"{m}: MONTH_PRED selects zero rows"
    assert chk.lo == chk.hi == m, f"{m}: MONTH_PRED selects months {chk.lo}..{chk.hi} per MONTH_EXPR"
    print(f"{m}: {chk.n:,.0f} rows, month predicate consistent")

## Phase 2 — every column, single months: absent rate (rows and dollars), NDV, domain values

In [ ]:
prof_cols = [(c, t) for c, t, _ in data_cols if not t.startswith(("array", "map", "struct"))]
rows = []
for m in PROFILE_MONTHS:
    for i0 in range(0, len(prof_cols), COL_CHUNK):
        chunk = prof_cols[i0:i0 + COL_CHUNK]
        sel = ["COUNT(*) AS n__", f"SUM({AMT}) AS usd__"]
        for i, (c, _) in enumerate(chunk):
            sel += [f"SUM(CASE WHEN {absent(c)} THEN 1 ELSE 0 END) AS a{i}",
                    f"SUM(CASE WHEN {absent(c)} THEN {AMT} ELSE 0 END) AS u{i}",
                    f"NDV({c}) AS d{i}"]
        r = q(f"SELECT {', '.join(sel)} FROM {TABLE} WHERE {month_pred(m)} AND {BASE_FILTER}")
        r = num(r, r.columns).iloc[0]
        for i, (c, t) in enumerate(chunk):
            rows.append(dict(month=m, column=c, type=t,
                             rows_absent_pct=100 * r[f"a{i}"] / r.n__,
                             usd_absent_pct=100 * r[f"u{i}"] / r.usd__ if r.usd__ else np.nan,
                             ndv=r[f"d{i}"]))
prof = pd.DataFrame(rows)
prof.to_csv(OUT_DIR / "p2_column_profile_long.csv", index=False)

role_of = {c: k for k, v in ROLES.items() for c in (v if isinstance(v, list) else [v]) if c}
wide = prof.pivot_table(index=["column", "type"], columns="month",
                        values=["rows_absent_pct", "usd_absent_pct", "ndv"])
wide.columns = [f"{a} [{b}]" for a, b in wide.columns]
wide = wide.reset_index()
wide.insert(2, "role", wide["column"].map(role_of).fillna(""))
wide = wide.sort_values(f"rows_absent_pct [{MONTH_MAX}]", ascending=False).round(2)
wide.to_csv(OUT_DIR / "p2_column_profile_wide.csv", index=False)
print("Absent rate by rows and dollars, NDV — first vs latest month (drift between the two is itself a finding):")
display(wide)

In [ ]:
lowcard = sorted(set(prof.loc[(prof.ndv <= LOWCARD_MAX) & (prof.column != ROLES["amount"]), "column"]))
dom = []
for m in PROFILE_MONTHS:
    for i0 in range(0, len(lowcard), 8):
        parts_sql = [f"SELECT '{c}' AS col, {lbl(c)} AS val, COUNT(*) AS n, SUM({AMT}) AS usd "
                     f"FROM {TABLE} WHERE {month_pred(m)} AND {BASE_FILTER} GROUP BY 2"
                     for c in lowcard[i0:i0 + 8]]
        d = num(q(" UNION ALL ".join(parts_sql)), ["n", "usd"])
        d["month"] = m
        dom.append(d)
dom = pd.concat(dom, ignore_index=True) if dom else pd.DataFrame(columns=["col", "val", "n", "usd", "month"])
g = dom.groupby(["month", "col"])
dom["rows%"] = (100 * dom.n / g.n.transform("sum")).round(2)
dom["usd%"] = (100 * dom.usd / g.usd.transform("sum")).round(2)
dom = dom.sort_values(["col", "month", "n"], ascending=[True, True, False])
dom.to_csv(OUT_DIR / "p2_domain_values.csv", index=False)

# Full tables for role columns; compact top-5 for the rest.
role_dims = [ROLES[k] for k in ("rail", "direction", "source_system", "merchant_flag", "status") if ROLES[k]]
for c in role_dims:
    if c in lowcard:
        print(f"\n=== {c} ({role_of[c]}) ===")
        display(dom[dom.col == c].pivot_table(index="val", columns="month", values=["rows%", "usd%"]).round(2)
                .sort_values(("rows%", MONTH_MAX), ascending=False))
    else:
        print(f"\n{c} ({role_of[c]}): NDV above {LOWCARD_MAX} — not a clean code column, inspect directly")
top5 = (dom[~dom.col.isin(role_dims)].groupby(["col", "month"])
        .apply(lambda x: " | ".join(f"{v} {r:.1f}%/{u:.1f}%$" for v, r, u in x.head(5)[["val", "rows%", "usd%"]].values))
        .unstack("month"))
print("\nOther low-cardinality columns — top 5 values as rows%/usd%:")
display(top5)

## Phase 3a — volume: month × rail × direction × source system

In [ ]:
vol = q(f"""
SELECT {MONTH_EXPR} AS month, {lbl(ROLES['rail'])} AS rail, {lbl(ROLES['direction'])} AS direction,
       {lbl(ROLES['source_system'])} AS source_system,
       COUNT(*) AS n, SUM({AMT}) AS usd,
       SUM(CASE WHEN {AMT_RAW} IS NULL THEN 1 ELSE 0 END) AS n_amt_null,   -- includes unparseable strings
       SUM(CASE WHEN {AMT_RAW} = 0 THEN 1 ELSE 0 END)     AS n_amt_zero,
       SUM(CASE WHEN {AMT_RAW} < 0 THEN 1 ELSE 0 END)     AS n_amt_neg,
       SUM(CASE WHEN {AMT_RAW} < 0 THEN {AMT} ELSE 0 END) AS usd_amt_neg
FROM {TABLE} WHERE {range_pred()} AND {BASE_FILTER}
GROUP BY 1, 2, 3, 4""", "p3a_volume_cube")
vol = num(vol, ["n", "usd", "n_amt_null", "n_amt_zero", "n_amt_neg", "usd_amt_neg"])

expected = pd.period_range(MONTH_MIN, MONTH_MAX, freq="M").strftime("%Y-%m")
gaps = sorted(set(expected) - set(vol.month))
extra = sorted(set(vol.month) - set(expected))
print(f"months present {vol.month.nunique()}/{len(expected)}; missing {gaps or 'none'}; outside range {extra or 'none'}")

print("\nRows (M) by month × rail:")
display((vol.pivot_table(index="month", columns="rail", values="n", aggfunc="sum", margins=True) / 1e6).round(3))
print("\nDollars ($B, |amount|) by month × rail:")
display((vol.pivot_table(index="month", columns="rail", values="usd", aggfunc="sum", margins=True) / 1e9).round(2))

rd = vol.groupby(["rail", "direction"])[["n", "usd", "n_amt_null", "n_amt_zero", "n_amt_neg", "usd_amt_neg"]].sum()
rd_out = pd.DataFrame({
    "rows_M": rd.n / 1e6, "usd_B": rd.usd / 1e9,
    "rows% of all": 100 * rd.n / rd.n.sum(), "usd% of all": 100 * rd.usd / rd.usd.sum(),
    "amt_null rows%": 100 * rd.n_amt_null / rd.n, "amt_zero rows%": 100 * rd.n_amt_zero / rd.n,
    "amt_neg rows%": 100 * rd.n_amt_neg / rd.n, "amt_neg usd%": 100 * rd.usd_amt_neg / rd.usd}).round(2)
print("\nRail × direction, with amount-sign profile (100% negative on one direction = sign encodes direction):")
display(rd_out)

print("\nSource system × rail — rows (M) and dollars ($B). A rail fed by >1 source is where cross-source duplicates can live:")
sr = vol.groupby(["source_system", "rail"])[["n", "usd"]].sum()
display(pd.concat([(sr.n / 1e6).unstack("rail").round(3), (sr.usd / 1e9).unstack("rail").round(2)],
                  axis=1, keys=["rows_M", "usd_B"]))

fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
(vol.pivot_table(index="month", columns="rail", values="n", aggfunc="sum") / 1e6).plot(ax=ax[0], marker=".")
(vol.pivot_table(index="month", columns="rail", values="usd", aggfunc="sum") / 1e9).plot(ax=ax[1], marker=".", legend=False)
ax[0].set_ylabel("rows (M)"); ax[1].set_ylabel("$B |amount|"); ax[0].set_title(f"{TABLE} — monthly volume by rail")
ax[0].legend(fontsize=8, ncol=4)
plt.xticks(rotation=90); plt.tight_layout(); fig.savefig(FIG_DIR / "p3a_volume_by_rail.png", dpi=130); plt.close(fig)

## Phase 3b — FI routing, FI name, counterparty name / id: coverage and overlap
One coverage cube; every margin and overlap below is derived from it in pandas.

In [ ]:
cube = q(f"""
SELECT {MONTH_EXPR} AS month, {lbl(ROLES['rail'])} AS rail, {lbl(ROLES['direction'])} AS direction,
       {lbl(ROLES['source_system'])} AS source_system,
       {rt_class(ROLES['fi_routing'])} AS fi_rt_class,
       {f"CASE WHEN {absent(ROLES['fi_name'])} THEN 0 ELSE 1 END" if ROLES['fi_name'] else "-1"} AS fi_name_present,
       {f"CASE WHEN {absent(ROLES['cpty_name'])} THEN 0 ELSE 1 END" if ROLES['cpty_name'] else "-1"} AS cpty_name_present,
       {f"CASE WHEN {absent(ROLES['cpty_id'])} THEN 0 ELSE 1 END" if ROLES['cpty_id'] else "-1"} AS cpty_id_present,
       COUNT(*) AS n, SUM({AMT}) AS usd
FROM {TABLE} WHERE {range_pred()} AND {BASE_FILTER}
GROUP BY 1, 2, 3, 4, 5, 6, 7, 8""", "p3b_coverage_cube")
cube = num(cube, ["n", "usd", "fi_name_present", "cpty_name_present", "cpty_id_present"])
assert abs(cube.n.sum() - vol.n.sum()) == 0, "coverage cube and volume cube disagree on row count"

RT = ["absent", "sentinel_zero", "malformed", "aba_checksum_fail", "bic_like", "aba_valid"]
cube["fi_usable"] = cube.fi_rt_class.isin(["aba_valid", "bic_like"])

print("FI routing class by rail × direction (rows% / usd%):")
display(pct_block(cube, ["rail", "direction"], {k: cube.fi_rt_class == k for k in RT}))

print("\nHeadline coverage by rail × direction — absent = field unusable for that purpose:")
display(pct_block(cube, ["rail", "direction"], {
    "FI routing unusable": ~cube.fi_usable,
    "FI name absent": cube.fi_name_present == 0,
    "cpty name absent": cube.cpty_name_present == 0,
    "cpty id absent": cube.cpty_id_present == 0}))

print("\nSame, by source system × rail — if duplicate sources differ in field population, that picks the keeper:")
display(pct_block(cube, ["source_system", "rail"], {
    "FI routing unusable": ~cube.fi_usable, "cpty name absent": cube.cpty_name_present == 0}))

ts = pct_block(cube, ["month", "rail"], {"FI usable": cube.fi_usable})
print("\nFI routing usable over time — usd% by month × rail (a step change = upstream feed change):")
display(ts["FI usable usd%"].unstack("rail"))
fig, ax = plt.subplots(figsize=(12, 5))
ts["FI usable usd%"].unstack("rail").plot(ax=ax, marker=".")
ax.set_ylabel("% of |amount| with usable routing"); ax.set_ylim(0, 105); ax.legend(fontsize=8, ncol=4)
plt.xticks(rotation=90); plt.tight_layout(); fig.savefig(FIG_DIR / "p3b_fi_usable_over_time.png", dpi=130); plt.close(fig)

nm0, fi0 = cube.cpty_name_present == 0, ~cube.fi_usable
print("\nOverlap: counterparty name × FI routing, by rail (rows% / usd% of the rail):")
display(pct_block(cube, ["rail"], {
    "name absent & FI unusable": nm0 & fi0, "name absent & FI usable": nm0 & ~fi0,
    "name present & FI unusable": ~nm0 & fi0}))
na = cube[nm0].groupby("rail")[["n", "usd"]].sum()
na_fi = cube[nm0 & fi0].groupby("rail")[["n", "usd"]].sum().reindex(na.index, fill_value=0)
print("\nP(FI unusable | name absent) by rail — near 100% means one upstream gap, not two:")
display(pd.DataFrame({"rows%": 100 * na_fi.n / na.n, "usd%": 100 * na_fi.usd / na.usd}).round(2))

## Phase 3c — beneficiary / counterparty address coverage

In [ ]:
if ROLES["address"]:
    meas = [f"SUM(CASE WHEN {absent(c)} THEN 0 ELSE 1 END) AS n_{c}, "
            f"SUM(CASE WHEN {absent(c)} THEN 0 ELSE {AMT} END) AS usd_{c}" for c in ROLES["address"]]
    addr = q(f"""
    SELECT {MONTH_EXPR} AS month, {lbl(ROLES['rail'])} AS rail, {lbl(ROLES['direction'])} AS direction,
           COUNT(*) AS n, SUM({AMT}) AS usd, {', '.join(meas)}
    FROM {TABLE} WHERE {range_pred()} AND {BASE_FILTER}
    GROUP BY 1, 2, 3""", "p3c_address_cube")
    addr = num(addr, [c for c in addr.columns if c.startswith(("n", "usd"))])
    for scope, frame in (("all months", addr), (MONTH_MAX, addr[addr.month == MONTH_MAX])):
        g = frame.groupby(["rail", "direction"]).sum(numeric_only=True)
        out = {"rows_M": g.n / 1e6}
        for c in ROLES["address"]:
            out[f"{c} rows%"] = 100 * g[f"n_{c}"] / g.n
            out[f"{c} usd%"] = 100 * g[f"usd_{c}"] / g.usd
        print(f"\nAddress field populated — {scope}:")
        display(pd.DataFrame(out).round(2))
else:
    print("No address columns in ROLES — geo block for counterparties stays on the locatability (neighbour) route")

## Phase 3d — counterparty-vs-merchant flag
Distribution by rail (all months); then for single months: is the flag a property of the counterparty
(constant per cpty) or of the transaction, and how it overlaps with counterparty degree.

In [ ]:
MF, CP = ROLES["merchant_flag"], ROLES["cpty_id"]
PNC = ROLES["cust_id"] or ROLES["acct_id"]
if MF:
    mfd = q(f"""
    SELECT {MONTH_EXPR} AS month, {lbl(ROLES['rail'])} AS rail, {lbl(ROLES['direction'])} AS direction,
           {lbl(MF)} AS mf, COUNT(*) AS n, SUM({AMT}) AS usd{f", NDV({CP}) AS ndv_cpty" if CP else ""}
    FROM {TABLE} WHERE {range_pred()} AND {BASE_FILTER}
    GROUP BY 1, 2, 3, 4""", "p3d_mflag_cube")
    mfd = num(mfd, ["n", "usd", "ndv_cpty"])
    print("Merchant flag by rail × direction (rows% / usd%):")
    display(pct_block(mfd, ["rail", "direction"], {v: mfd.mf == v for v in sorted(mfd.mf.unique())}))
    print("\nMerchant flag share of dollars over time:")
    display((100 * mfd.pivot_table(index="month", columns="mf", values="usd", aggfunc="sum")
             .pipe(lambda x: x.div(x.sum(axis=1), axis=0))).round(2))

    if CP and PNC:
        deg_bucket = """CASE WHEN d.deg = 1 THEN 'a 1' WHEN d.deg < 5 THEN 'b 2-4' WHEN d.deg < 15 THEN 'c 5-14'
                             WHEN d.deg < 100 THEN 'd 15-99' WHEN d.deg < 1000 THEN 'e 100-999'
                             WHEN d.deg < 10000 THEN 'f 1k-10k' ELSE 'g 10k+' END"""
        for m in PROFILE_MONTHS:
            cte = f"""
            WITH e AS (SELECT {CP} AS cpty, {lbl(MF)} AS mf, {lbl(ROLES['rail'])} AS rail, {PNC} AS pnc,
                              {AMT} AS usd{f", {ROLES['cpty_name']} AS nm" if ROLES['cpty_name'] else ""}
                       FROM {TABLE} WHERE {month_pred(m)} AND {BASE_FILTER} AND NOT {absent(CP)}),
                 d AS (SELECT cpty, COUNT(DISTINCT pnc) AS deg, MIN(mf) AS mf_min, MAX(mf) AS mf_max
                       FROM e GROUP BY cpty)"""
            md = num(q(f"""{cte}
            SELECT e.mf, e.rail, {deg_bucket} AS deg_bucket,
                   CASE WHEN d.mf_min <> d.mf_max THEN 1 ELSE 0 END AS mf_varies,
                   COUNT(*) AS n, SUM(e.usd) AS usd, COUNT(DISTINCT e.cpty) AS n_cpty
            FROM e JOIN d ON e.cpty = d.cpty GROUP BY 1, 2, 3, 4""", f"p3d_mflag_degree_{m}"),
                     ["n", "usd", "n_cpty", "mf_varies"])
            print(f"\n[{m}] flag consistency — share of rows/dollars whose counterparty carries >1 flag value this month:")
            display(pct_block(md, ["rail"], {"cpty flag varies": md.mf_varies == 1}))
            print(f"[{m}] flag × counterparty degree (distinct PNC {'customers' if ROLES['cust_id'] else 'accounts'}; "
                  f"15+ ≈ the median-multiple hub line when median degree is 1):")
            display(pct_block(md, ["deg_bucket"], {v: md.mf == v for v in sorted(md.mf.unique())}))
            top = q(f"""{cte}
            SELECT d.cpty, d.deg, d.mf_min, d.mf_max, COUNT(*) AS n, SUM(e.usd) AS usd,
                   MIN(e.rail) AS rail_min, MAX(e.rail) AS rail_max
                   {", MAX(e.nm) AS cpty_name" if ROLES['cpty_name'] else ""}
            FROM e JOIN d ON e.cpty = d.cpty
            GROUP BY d.cpty, d.deg, d.mf_min, d.mf_max ORDER BY d.deg DESC LIMIT 50""", f"p3d_top_degree_{m}")
            print(f"[{m}] top 50 counterparties by degree:")
            display(num(top, ["deg", "n", "usd"]))
    else:
        print("cpty_id or PNC-side id not set — degree overlap skipped")
else:
    print("merchant_flag not set — skipped")

## Phase 3e — trx_id structure, within-source duplicate ratio, timestamp granularity
Groundwork for the 0b duplicate matcher: which sources share a rail, what their ids look like,
and whether a minutes-wide match window is even possible.

In [ ]:
TID, TS, SRC = ROLES["trx_id"], ROLES["txn_ts"], ROLES["source_system"]
if TID:
    s = f"TRIM(CAST({TID} AS STRING))"
    shape = (f"regexp_replace(regexp_replace(regexp_replace(regexp_replace({s}, "
             f"'[0-9]', '9'), '[A-Za-z]', 'A'), '9+', '9'), 'A+', 'A')")
    ts_cast = f"CAST({TS} AS TIMESTAMP)" if TS else None
    for m in PROFILE_MONTHS:
        dup = num(q(f"""
        SELECT {lbl(SRC)} AS source_system, {lbl(ROLES['rail'])} AS rail,
               COUNT(*) AS n, SUM({AMT}) AS usd,
               SUM(CASE WHEN {absent(TID)} THEN 1 ELSE 0 END) AS n_tid_absent,
               COUNT(DISTINCT {TID}) AS n_tid_distinct
               {f''', SUM(CASE WHEN {ts_cast} IS NULL THEN 1 ELSE 0 END) AS n_ts_null,
                   SUM(CASE WHEN hour({ts_cast}) = 0 AND minute({ts_cast}) = 0 AND second({ts_cast}) = 0
                            THEN 1 ELSE 0 END) AS n_ts_midnight''' if TS else ""}
        FROM {TABLE} WHERE {month_pred(m)} AND {BASE_FILTER}
        GROUP BY 1, 2""", f"p3e_tid_dup_{m}"), ["n", "usd", "n_tid_absent", "n_tid_distinct",
                                                  "n_ts_null", "n_ts_midnight"])
        dup["rows_per_tid"] = (dup.n - dup.n_tid_absent) / dup.n_tid_distinct.replace(0, np.nan)
        dup["tid_absent%"] = 100 * dup.n_tid_absent / dup.n
        if TS:
            dup["ts_null%"] = 100 * dup.n_ts_null / dup.n
            dup["ts_midnight%"] = 100 * dup.n_ts_midnight / dup.n   # ~100% => date-only; same-day window is the floor
        print(f"\n[{m}] within-source trx_id reuse (rows_per_tid > 1 = id repeats inside a source) and timestamp grain:")
        display(dup.drop(columns=[c for c in ("n_tid_absent", "n_tid_distinct", "n_ts_null", "n_ts_midnight")
                                  if c in dup.columns]).round(3))

        shp = num(q(f"""
        SELECT {lbl(SRC)} AS source_system, {lbl(ROLES['rail'])} AS rail,
               LENGTH({s}) AS len, {shape} AS shape,
               CASE WHEN regexp_like(substr({s}, 1, 3), '^[A-Za-z]{{3}}$') THEN upper(substr({s}, 1, 3)) END AS alpha_prefix,
               COUNT(*) AS n, SUM({AMT}) AS usd
        FROM {TABLE} WHERE {month_pred(m)} AND {BASE_FILTER} AND NOT {absent(TID)}
        GROUP BY 1, 2, 3, 4, 5""", f"p3e_tid_shape_{m}"), ["n", "usd", "len"])
        shp["rows%"] = (100 * shp.n / shp.groupby(["source_system", "rail"]).n.transform("sum")).round(2)
        shp = shp.sort_values(["source_system", "rail", "n"], ascending=[True, True, False])
        print(f"[{m}] trx_id shapes (digits->9, letters->A, runs collapsed), top 8 per source × rail:")
        display(shp.groupby(["source_system", "rail"]).head(8))
else:
    print("trx_id not set — skipped")

## What to bring back
Tables inline above; charts in `figs/`; every result as CSV next to its SQL.
Next: **0b** (PySpark) — node-key parity against the Neo4j ingestion logic, on-us two-sided rows vs
cross-source duplicates, and the candidate-duplicate matcher on parties + amount + time window.